# Analyze Account Data Sources

We have three sources for account and account holder data:
1. Direct download of the account data
2. Download through the power BI app
3. Inferred from transaction data

We need to examine what are the differences in different source and which information
we take from which of the sources.

## Packages and options

In [14]:
# add the parent directory to the sys.path
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd

In [15]:
fn_direct = "../data/source/automatic/eutl_accounts.csv"
fn_bi = "../data/source/manual/accounts.xlsx"
fn_trans = "../data/source/automatic/eutl_transactions.csv"

## Get data

Account data from direct download:

In [31]:
df_acc_direct = (
    pd.read_csv(fn_direct)
    .assign(
        account_id=lambda df: df["REGISTRY_CODE"] + "_" + df["ACCOUNT_IDENTIFIER"].astype(str),
    )
)
map_registry_names = df_acc_direct.set_index("REGISTRY_NAME")["REGISTRY_CODE"].to_dict()
df_acc_direct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47685 entries, 0 to 47684
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ACCOUNT_IDENTIFIER    47685 non-null  int64 
 1   REGISTRY_CODE         47685 non-null  object
 2   REGISTRY_NAME         47685 non-null  object
 3   ACCOUNT_NAME          47685 non-null  object
 4   ACCOUNT_TYPE          47685 non-null  object
 5   ETS_ACCOUNT_TYPE      28367 non-null  object
 6   FULL_TYPE             47656 non-null  object
 7   OPEN_DATE             47649 non-null  object
 8   END_OF_VALIDITY_DATE  47685 non-null  object
 9   IS_CLOSURE_PENDING    47685 non-null  object
 10  SNAPSHOT_DATE         47685 non-null  object
 11  account_id            47685 non-null  object
dtypes: int64(1), object(11)
memory usage: 4.4+ MB


In [42]:
df_acc_direct.ACCOUNT_TYPE.unique()

array(['Operator Holding Account', 'Person Account in National Registry',
       'Holding Account', 'Voluntary Cancellation Account (Type 3)',
       'Retirement Account',
       'tCER Replacement Account for Expiry (Type 1)',
       'lCER Replacement Account for Expiry (Type 1)',
       'Net Source Cancellation Account (Type 1)',
       'Non-Kyoto Account Type',
       'Non-compliance Cancellation Account (Type 2)',
       'Mandatory (Cancellation Account (Type 5)',
       'lCER Replacement Account for Non-submission of Certification Report (Type 3)',
       'lCER Replacement Account for Reversal in Storage (Type 2)',
       'Excess Issuance Cancellation Account (Type 4)',
       'Previous Period Surplus Reserve Account (PPSR)',
       'Ambition Increase Cancellation Account (Type 8)',
       'Article 3.7ter Cancellation Account (Type 7)'], dtype=object)

In [41]:
df_acc_direct.ETS_ACCOUNT_TYPE.unique()

array([nan, 'AAU Deposit Account', 'Operator Holding Account',
       'Person Holding Account', 'Trading Account',
       'National Allowance Holding Account', 'Aircraft Operator Account',
       'Verifier Account', 'Gateway Deposit Account',
       'Aviation Surrender Set-Aside Account', 'Central Clearing Account',
       'Union Allowance Deletion Account', 'Auction Delivery Account',
       'Total Quantity Account', 'Aviation Total Quantity Account',
       'Allocation Account', 'New Entrant Reserve Account',
       'Auction Account', 'Aviation Auction Account',
       'Aviation Allocation Account', 'International Credit Account',
       'Credit Exchange Account', 'Maritime Operator Holding Account',
       'ESD AAU Deposit Account', 'ETS Central Clearing Account for CP2',
       'ETS AAU Deposit Account', 'ESD Compliance Account',
       'EU AAU Account', 'ESD Central Clearing Account',
       'AEA Deletion Account', 'AEA Total quantity Account'], dtype=object)

Power BI data

In [21]:
df_acc_bi = (
    pd.read_excel(fn_bi, skipfooter=2)
    .rename(columns={"..1": "registry_id"})
    .drop(columns=".")
    .assign(
        account_id=lambda df: df["registry_id"] + "_" + df["Account Identifier"].astype(str),
    )
)
df_acc_bi.info()

c:\GIT\eutl_scraper_v2\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47389 entries, 0 to 47388
Data columns (total 15 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   Account Identifier                                   47389 non-null  int64  
 1   National Administrator                               47389 non-null  object 
 2   Account Type                                         47389 non-null  object 
 3   Account Holder Name                                  47389 non-null  object 
 4   Account Name                                         47389 non-null  object 
 5   Installation/Aircraft Operator/Maritime Operator ID  22158 non-null  float64
 6   Company Registration No                              42781 non-null  object 
 7   Main Address Line                                    47380 non-null  object 
 8   City                                                 47380 non-nul

Transaction data

In [39]:
def assign_account_id(row) -> str:
    if pd.notnull(row["ACCOUNT_IDENTIFIER"]):
        return (
            map_registry_names.get(row["REGISTRY_NAME"], "UNKOWN") 
            + "_" + str(int(row["ACCOUNT_IDENTIFIER"]))
        )


df_acc_trans = pd.read_csv("../data/source/automatic/eutl_transactions.csv")
# extract account involved in transactions
lst_df = []
for prefix in ["TRANSFERRING", "ACQUIRING"]:
    cols = [c for c in df_acc_trans.columns if c.startswith(prefix)]
    df_ = df_acc_trans[cols].copy()
    cols = [c.replace(f"{prefix}_", "") for c in cols]
    df_.columns = cols
    lst_df.append(df_)
df_acc_trans = (
    pd.concat(lst_df, axis=0)
    .drop_duplicates()
    .reset_index(drop=True)
    .assign(
        registry_id=lambda df: df["REGISTRY_NAME"].map(map_registry_names),
        account_id=lambda df: df.apply(assign_account_id, axis=1)
    )
)
df_acc_trans.info()

C:\Users\abrell\AppData\Local\Temp\ipykernel_9928\3526116671.py:9: DtypeWarning: Columns (20,22,23,24,25,26,27,28,29,47,49,50,51,52,53,54,55,56,60,62,65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_acc_trans = pd.read_csv("../data/source/automatic/eutl_transactions.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35113 entries, 0 to 35112
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   REGISTRY_NAME                               35113 non-null  object 
 1   ACCOUNT_TYPE1                               35083 non-null  float64
 2   ACCOUNT_TYPE2                               35113 non-null  object 
 3   ACCOUNT_TYPE3                               35113 non-null  object 
 4   ACCOUNT_OPEN_DT                             34304 non-null  object 
 5   ACCOUNT_END_OF_VALIDITY                     21167 non-null  object 
 6   ACCOUNT_NAME                                34323 non-null  object 
 7   ACCOUNT_IDENTIFIER                          35083 non-null  float64
 8   ACCOUNT_HOLDER                              34323 non-null  object 
 9   ACCOUNT_HOLDER_ADDRESS1                     34322 non-null  object 
 10  ACCOUNT_HO

{}

In [ ]:
df = df_acc_trans.copy()

df.apply(assign_account_id, axis=1)

0           DE_1914
1            GB_901
2            NL_286
3           DE_2138
4           GB_1683
            ...    
35108       GB_1575
35109    ES_5017407
35110    CZ_5024111
35111        GR_160
35112    SI_5024066
Length: 35113, dtype: object

## How do direct downloads compare against power bi accounts?

Overall we observe:

1. Only direct downloads provide information on dates
2. Only the PowerBi downloads provide information on the account holder 

In [19]:
acc_only_direct = set(df_acc_direct["account_id"]) - set(df_acc_bi["account_id"])
acc_only_bi = set(df_acc_bi["account_id"]) - set(df_acc_direct["account_id"])
print(f"Accounts only in direct download: {len(acc_only_direct)}")
print(f"Accounts only in BI download: {len(acc_only_bi)}")

Accounts only in direct download: 296
Accounts only in BI download: 0


### Account types

Account types are aggregated by groups and more disaggregated in the Power BI app.

In [44]:
df_ = df_acc_direct.copy()
df_[df_["ETS_ACCOUNT_TYPE"].notnull()][["ACCOUNT_TYPE", "ETS_ACCOUNT_TYPE"]].drop_duplicates()

,ACCOUNT_TYPE,ETS_ACCOUNT_TYPE
4875,Holding Account,AAU Deposit Account
12343,Holding Account,Operator Holding Account
12447,Holding Account,Person Holding Account
12492,Holding Account,Trading Account
12589,Holding Account,National Allowance Holding Account
15176,Holding Account,Aircraft Operator Account
16052,Non-Kyoto Account Type,Verifier Account
18756,Holding Account,Gateway Deposit Account
26253,Holding Account,Aviation Surrender Set-Aside Account
28749,Holding Account,Central Clearing Account


In [43]:
df_acc = df_acc_direct.merge(df_acc_bi, on="account_id", how="outer")
direct, bi = "FULL_TYPE", "Account Type"
df_ = df_acc[(df_acc[direct].str.strip() != df_acc[bi].str.strip())]
print(f"{len(df_)} differing account types between direct downloads and Power BI")
df_ = (
    df_[[direct, bi]]
    .sort_values(by=direct)
    .drop_duplicates()
)
df_


300 differing account types between direct downloads and Power BI


,FULL_TYPE,Account Type
14029,AEA Deletion Account,NaN
14028,AEA Total quantity Account,NaN
14138,ESD Compliance Account,NaN
2495,Former Operator Holding Account,NaN
39113,Maritime Operator Holding Account,NaN
3846,Operator Holding Account,NaN
36745,Party Holding Account,NaN
2500,Person Account in National Registry,NaN
2505,Retirement Account,NaN
35673,Trading Account,NaN


In [ ]:
to_check = {
    "Account Name": "ACCOUNT_NAME",
    
}

for right, left in to_check.items():
    df_ = df_acc[(df_acc[left].str.strip() != df_acc[right].str.strip())]
    if not df_.empty:
        df_ = df_[[left, right]].sort_values(by=left).drop_duplicates()
        break
df_acc[[left, right]].drop_duplicates().sort_values(by=left)
df_

,ACCOUNT_NAME,Account Name
1840,8160 MARI KOKAKO,NaN
13297,A/S GLOBAL RISK MANAGEMENT LTD. HOLDING,A/S Global Risk Management Ltd. Holding
28091,A2A S.p.A.,A2A S.P.A.
34787,A2A Trading,a2a Trading
22621,ABN AMRO BANK N.v.,ABN AMRO BANK N.V.
...,...,...
33248,stabilimento di Brindisi,Stabilimento di Brindisi
14257,ubr logistik,UBR Logistik
34803,veronagest,Veronagest
5644,vertus energiehandel gmbh,Vertus Energiehandel Gmbh
